In [ ]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

2.1.0+cu121
True


In [1]:
import os
import requests
import zipfile
from tqdm import tqdm
from pathlib import Path

# Directory to store downloaded and extracted data
DATA_DIR = Path("./mimic_textbooks")

# Step 1: Download and extract the dataset zip file
def download_and_extract_zip(url, extract_to=DATA_DIR):
    # Ensure the directory exists
    extract_to.mkdir(parents=True, exist_ok=True)

    # Download the zip file
    zip_path = extract_to / "textbooks.zip"
    print("Downloading dataset...")
    response = requests.get(url, stream=True)
    with open(zip_path, "wb") as file:
        for chunk in tqdm(response.iter_content(chunk_size=1024), unit='KB'):
            if chunk:
                file.write(chunk)

    # Extract the zip file
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_to)
    print("Dataset downloaded and extracted.")

# Step 2: Load and process text files
def load_text_files(directory):
    text_data = []
    file_paths = []

    # Use pathlib.Path to dynamically list all .txt files in directory
    for filepath in Path(directory).rglob("*.txt"):
        with open(filepath, "r", encoding="utf-8") as file:
            text = file.read()
            text_data.append(text)
            file_paths.append(filepath)

    return text_data, file_paths

# Execute the download, extraction, and loading setup
def setup_environment():
    # Zip file URL
    dataset_url = "https://www.dropbox.com/scl/fi/54p9kkx5n93bffyx08eba/textbooks.zip?rlkey=2y2c5x8y0uncnddichn9cmd7n&st=m290nmkk&dl=1"

    # Step 1: Download and extract data files
    download_and_extract_zip(dataset_url)

    # Step 2: Load text files from the extracted dataset
    text_files_directory = DATA_DIR / "textbooks/en"
    texts, paths = load_text_files(text_files_directory)

    print("Processing complete. Loaded {} documents.".format(len(paths)))
    return texts, paths

# Run the setup
texts, document_paths = setup_environment()


88121KB [00:00, 99114.25KB/s] 


Extracting dataset...
Dataset downloaded and extracted.
Processing complete. Loaded 18 documents.


In [2]:
document_paths

[PosixPath('mimic_textbooks/textbooks/en/Histology_Ross.txt'),
 PosixPath('mimic_textbooks/textbooks/en/InternalMed_Harrison.txt'),
 PosixPath('mimic_textbooks/textbooks/en/Neurology_Adams.txt'),
 PosixPath('mimic_textbooks/textbooks/en/Anatomy_Gray.txt'),
 PosixPath('mimic_textbooks/textbooks/en/Immunology_Janeway.txt'),
 PosixPath('mimic_textbooks/textbooks/en/Gynecology_Novak.txt'),
 PosixPath('mimic_textbooks/textbooks/en/Pharmacology_Katzung.txt'),
 PosixPath('mimic_textbooks/textbooks/en/Psichiatry_DSM-5.txt'),
 PosixPath('mimic_textbooks/textbooks/en/Pediatrics_Nelson.txt'),
 PosixPath('mimic_textbooks/textbooks/en/First_Aid_Step1.txt'),
 PosixPath('mimic_textbooks/textbooks/en/Physiology_Levy.txt'),
 PosixPath('mimic_textbooks/textbooks/en/Pathology_Robbins.txt'),
 PosixPath('mimic_textbooks/textbooks/en/First_Aid_Step2.txt'),
 PosixPath('mimic_textbooks/textbooks/en/Cell_Biology_Alberts.txt'),
 PosixPath('mimic_textbooks/textbooks/en/Biochemistry_Lippincott.txt'),
 PosixPath('

In [3]:
!pip install requests tqdm faiss-cpu transformers tensorflow sentence-transformers textblob gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 87.3 MB/s eta 0:00:00


In [4]:
import os
import requests
import zipfile
from pathlib import Path
from tqdm import tqdm

# Directory to store downloaded and extracted data
DATA_DIR = Path("./mimic_textbooks")

# Download and extract the dataset zip file
def download_and_extract_zip(url, extract_to=DATA_DIR):
    # Ensure the directory exists
    extract_to.mkdir(parents=True, exist_ok=True)

    # Download the zip file
    zip_path = extract_to / "textbooks.zip"
    print("Downloading dataset...")
    response = requests.get(url, stream=True)
    with open(zip_path, "wb") as file:
        for chunk in tqdm(response.iter_content(chunk_size=1024), unit='KB'):
            if chunk:
                file.write(chunk)

    # Extract the zip file
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_to)
    print("Dataset downloaded and extracted.")

# Download and extract textbooks
dataset_url = "https://www.dropbox.com/scl/fi/54p9kkx5n93bffyx08eba/textbooks.zip?rlkey=2y2c5x8y0uncnddichn9cmd7n&st=m290nmkk&dl=1"
download_and_extract_zip(dataset_url)


88121KB [00:00, 95127.08KB/s]


Extracting dataset...
Dataset downloaded and extracted.


In [5]:
import re
from gensim.utils import simple_preprocess
from textblob import TextBlob

# Load text files
def load_text_files(directory):
    texts = []
    for file_path in Path(directory).glob("*.txt"):
        with open(file_path, "r", encoding="utf-8") as file:
            texts.append(file.read())
    return texts

# Cleaning and preprocessing function
def clean_and_tokenize(text):
    # Basic regex cleaning
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    text = text.lower()  # Lowercase all text
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters

    # Tokenize with gensim
    tokens = simple_preprocess(text)
    return ' '.join(tokens)

# Spell correction
def correct_spelling(text):
    return str(TextBlob(text).correct())

# Chunk text into fixed-size chunks
def chunk_text(text, chunk_size=200):
    words = text.split()
    return [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

# Load, clean, correct, and chunk documents
documents = load_text_files(DATA_DIR / "textbooks/en")
cleaned_documents = [clean_and_tokenize(doc) for doc in documents]
# corrected_documents = [correct_spelling(doc) for doc in cleaned_documents]
chunked_documents = []
for doc in cleaned_documents:
    chunked_documents.extend(chunk_text(doc))

print(f"Total document chunks created: {len(chunked_documents)}")


Total document chunks created: 60061


In [8]:
from sentence_transformers import SentenceTransformer
import tensorflow as tf # Keep for GPU device check
import numpy as np
from tqdm import tqdm # Import tqdm for progress bar

# Verify that TensorFlow detects the GPU
print("Available devices:", tf.config.list_physical_devices('GPU'))

# Load the model using SentenceTransformer, which handles both tokenizer and model
# It will automatically detect if TF or PyTorch should be used based on available installations.
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Function to generate embeddings for all chunks
def get_embeddings_in_batch(texts, batch_size=128):
    # SentenceTransformer's encode method is optimized for this
    # It handles tokenization, batching, and device placement internally.
    embeddings = model.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)
    return embeddings

# Generate embeddings for all document chunks in batches
embeddings = get_embeddings_in_batch(chunked_documents, batch_size=128)
print(f"Generated embeddings for {len(embeddings)} document chunks.")

Available devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/470 [00:00<?, ?it/s]

Generated embeddings for 60061 document chunks.


In [9]:
import faiss
import numpy as np

# Define the dimension of embeddings
dimension = 384  # Embedding size from MiniLM model
index = faiss.IndexFlatL2(dimension)

# Convert embeddings to NumPy array for FAISS
embedding_matrix = np.array([embedding.flatten() for embedding in embeddings]).astype('float32')

# Add embeddings to FAISS index
index.add(embedding_matrix)
print(f"Total embeddings indexed: {index.ntotal}")

Total embeddings indexed: 60061


In [16]:
# Example query for testing

query_text = "medicines for asthma?"
query_embedding = model.encode(query_text, convert_to_numpy=True) # Use model.encode directly
query_embedding = np.array(query_embedding).reshape(1, -1).astype('float32')

# Search FAISS for the most similar documents
k = 5  # Number of closest documents to retrieve
distances, indices = index.search(query_embedding, k)

# Retrieve and print the most similar chunks
print("Top similar document chunks:")
for idx in indices[0]:
    print(chunked_documents[idx], end="\n\n")

Top similar document chunks:
plus salmeterol versus fluticasone alone engl med stempel da et al safety of adding salmeterol to fluticasone propionate in children with asthma engl med barnes pj theophylline am respir crit care med rabe kf roflumilast for the treatment of chronic obstructive pulmonary disease expert rev respir med guevara et al inhaled corticosteroids versus sodium cromoglycate in children and adults with asthma cochrane database syst rev cd barnes how corticosteroids control inflammation quintiles prize lecture br pharmacol beasley et al combination inhaler as reliever therapy solution for intermittent and mild asthma allergy clin immunol boushey ha et al daily versus asneeded corticosteroids for mild persistent asthma engl med suissa et al lowdose inhaled corticosteroids and the prevention of death from asthma engl med damato et al anticholinergic drugs in asthma therapy curr opin pulm med lee am jacoby db fryer ad selective muscarinic receptor antagonists for airway d